# Prototype: Erste Exploration der Rohdaten

Dieses Notebook dient als erster Kontakt mit den NYC Parking Violations Rohdaten. Ziel ist es, die für unsere Fragestellungen relevanten Felder zu verstehen und mögliche Datenqualitätsprobleme zu erkennen, bevor das Pre-processing durchgeführt wird.

## Fragestellungen

1. Welche Parking-Violation-Typen kommen in FY2023–FY2025 am häufigsten vor, und wie verändern sie sich über die Fiskaljahre?
2. Gibt es zeitliche Muster bei Parking Violations nach Monat, Wochentag und Tageszeit?

## Ziel

- Rohdaten aus HDFS laden
- 1%-Sample ziehen für schnelle Exploration ohne lange Wartezeiten
- Struktur des Datensatzes verstehen
- Erste Zeilen anschauen
- Relevante Felder für Fragestellung 1 prüfen: `Violation Code`, `Violation Description`
- Relevante Felder für Fragestellung 2 prüfen: `Issue Date`, `Violation Time`
- Eindeutigkeit des Primary Keys (`Summons Number`) prüfen
- Verteilung pro Fiskaljahr prüfen

Die Erkenntnisse aus diesem Notebook fliessen ins Pre-processing (`src/1_Pre_Processing/5.0_clean_parking_violations.ipynb`) ein.

In [1]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, lit, sum as spark_sum

spark = SparkSession.builder \
    .appName("BDLC_Parking_Violations_RawPrototype") \
    .master("spark://bdlc-012.bdlc.ls.eee.intern:7077") \
    .config("spark.executor.cores", "4") \
    .config("spark.executor.memory", "15g") \
    .config("spark.cores.max", "12") \
    .getOrCreate()

spark

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/05/30 11:32:47 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
26/05/30 11:32:47 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.


In [2]:
# Rohdaten laden
raw_paths = {
    2023: "hdfs:///parking_violations/raw/2023/parking_violations_2023.csv",
    2024: "hdfs:///parking_violations/raw/2024/parking_violations_2024.csv",
    2025: "hdfs:///parking_violations/raw/2025/parking_violations_2025.csv",
}

dfs = []
for fiscal_year, path in raw_paths.items():
    df_year = spark.read.csv(path, header=True, inferSchema=False) \
        .withColumn("Fiscal Year", lit(fiscal_year))
    dfs.append(df_year)

df_raw = dfs[0]
for df_next in dfs[1:]:
    df_raw = df_raw.unionByName(df_next)

print(f"Zeilen: {df_raw.count():,}")
print(f"Spalten: {len(df_raw.columns)}")

[Stage 3:======================================================>  (73 + 3) / 76]

Zeilen: 54,223,582
Spalten: 44


In [3]:
# 1%-Sample ziehen
sample_raw = df_raw.sample(fraction=0.01, seed=42)
print(f"Sample-Grösse: {sample_raw.count():,} Zeilen")

26/05/30 11:33:04 WARN GarbageCollectionMetrics: To enable non-built-in garbage collector(s) List(G1 Concurrent GC), users should configure it(them) to spark.eventLog.gcMetrics.youngGenerationGarbageCollectors or spark.eventLog.gcMetrics.oldGenerationGarbageCollectors
[Stage 6:=======================================================> (74 + 2) / 76]

Sample-Grösse: 542,412 Zeilen


In [4]:
# Erste Zeilen anschauen — relevante Felder
sample_raw.select(
    "Fiscal Year",
    "Summons Number",
    "Issue Date",
    "Violation Time",
    "Violation Code",
    "Violation Description"
).show(10, truncate=False)

+-----------+--------------+----------+--------------+--------------+---------------------+
|Fiscal Year|Summons Number|Issue Date|Violation Time|Violation Code|Violation Description|
+-----------+--------------+----------+--------------+--------------+---------------------+
|2023       |1413271790    |05/03/2022|0144A         |17            |NULL                 |
|2023       |1438931517    |07/02/2022|0937P         |46            |NULL                 |
|2023       |1480582840    |06/28/2022|0218A         |46            |NULL                 |
|2023       |1453978124    |06/15/2022|0826A         |40            |NULL                 |
|2023       |1471480896    |06/30/2022|1115P         |98            |NULL                 |
|2023       |1471537778    |06/10/2022|0710P         |98            |NULL                 |
|2023       |1471561100    |07/03/2022|0110A         |19            |NULL                 |
|2023       |1482401307    |06/25/2022|0503P         |14            |NULL       

In [5]:
# Null-Werte der relevanten Felder prüfen
relevant_columns = [
    "Summons Number",
    "Issue Date",
    "Violation Time",
    "Violation Code",
    "Violation Description"
]

sample_raw.select([
    spark_sum(col(c).isNull().cast("int")).alias(c)
    for c in relevant_columns
]).show(truncate=False)

[Stage 10:======================================================> (74 + 2) / 76]

+--------------+----------+--------------+--------------+---------------------+
|Summons Number|Issue Date|Violation Time|Violation Code|Violation Description|
+--------------+----------+--------------+--------------+---------------------+
|0             |28        |14            |0             |10626                |
+--------------+----------+--------------+--------------+---------------------+



In [6]:
# Eindeutigkeit des Primary Keys prüfen (Summons Number)
total = sample_raw.count()
distinct = sample_raw.select("Summons Number").distinct().count()

print(f"Gesamtzeilen:            {total:,}")
print(f"Distinct Summons Number: {distinct:,}")
print(f"Duplikate:               {total - distinct:,}")

[Stage 16:=======================================================>(75 + 1) / 76]

Gesamtzeilen:            542,412
Distinct Summons Number: 542,009
Duplikate:               403


In [7]:
# Verteilung pro Fiskaljahr
sample_raw.groupBy("Fiscal Year").count().orderBy("Fiscal Year").show()

[Stage 22:=======================================================>(75 + 1) / 76]

+-----------+------+
|Fiscal Year| count|
+-----------+------+
|       2023|216095|
|       2024|161132|
|       2025|165185|
+-----------+------+



In [8]:
# Fragestellung 1: Violation Code und Description anschauen
from pyspark.sql.functions import desc

sample_raw.groupBy("Violation Code", "Violation Description").count() \
    .orderBy(desc("count")) \
    .show(20, truncate=False)

[Stage 25:======================================================> (74 + 2) / 76]

+--------------+------------------------------+------+
|Violation Code|Violation Description         |count |
+--------------+------------------------------+------+
|36            |PHTO SCHOOL ZN SPEED VIOLATION|185094|
|21            |21-No Parking (street clean)  |46071 |
|38            |38-Failure to Dsplay Meter Rec|38176 |
|14            |14-No Standing                |25668 |
|5             |BUS LANE VIOLATION            |22978 |
|7             |FAILURE TO STOP AT RED LIGHT  |22740 |
|40            |40-Fire Hydrant               |19311 |
|21            |No Parking Street Cleaning    |16726 |
|71            |71A-Insp Sticker Expired (NYS)|16156 |
|20            |20A-No Parking (Non-COM)      |14546 |
|70            |70A-Reg. Sticker Expired (NYS)|10629 |
|37            |37-Expired Parking Meter      |8825  |
|31            |31-No Stand (Com. Mtr. Zone)  |8353  |
|69            |69-Fail to Dsp Prking Mtr Rcpt|7649  |
|19            |19-No Stand (bus stop)        |7466  |
|16       

In [9]:
# Fragestellung 2: Issue Date anschauen 
from pyspark.sql.functions import to_date, year

sample_raw = sample_raw.withColumn(
    "issue_date_parsed",
    to_date(col("Issue Date"), "MM/dd/yyyy")
).withColumn(
    "issue_year",
    year(col("issue_date_parsed"))
)

sample_raw.groupBy("issue_year").count().orderBy("issue_year").show(30)

[Stage 28:=======================================================>(75 + 1) / 76]

+----------+------+
|issue_year| count|
+----------+------+
|      NULL|    28|
|      2000|     2|
|      2012|     1|
|      2020|     3|
|      2021|     6|
|      2022| 92101|
|      2023|208931|
|      2024|163761|
|      2025| 77572|
|      2026|     3|
|      2027|     1|
|      2028|     1|
|      2029|     1|
|      2052|     1|
+----------+------+



In [10]:
# Fragestellung 2: Violation Time anschauen
sample_raw.select("Violation Time").distinct().show(20, truncate=False)

[Stage 31:=====================================================>  (73 + 3) / 76]

+--------------+
|Violation Time|
+--------------+
|0756A         |
|0840P         |
|0449P         |
|1027A         |
|1024A         |
|0713A         |
|0126A         |
|0448A         |
|1010P         |
|0831A         |
|1042A         |
|0135P         |
|0911P         |
|0435P         |
|0519P         |
|0113A         |
|0525P         |
|0840A         |
|0648P         |
|0128P         |
+--------------+
only showing top 20 rows



## Erkenntnisse aus der Rohdaten-Exploration

Die Exploration des 1%-Samples der Rohdaten (542'412 Zeilen) zeigt folgende Auffälligkeiten:

1. **`Violation Description` inkonsistent:** 10'626 Einträge im Sample haben keine Beschreibung. Die vorhandenen Beschreibungen sind inkonsistent d.h. verschiedene Formulierungen für denselben Violation Code sind sichtbar (z.B. Code 21). Eine einheitliche, offizielle Beschreibung pro Code wäre für Analysen sinnvoll.

2. **`Violation Time` im ungewöhnlichen 12-Stunden-Format:** Das Zeitfeld enthält Werte wie `0834A` (08:34 AM) oder `0215P` (14:15 PM). Vereinzelt gibt es ungültige Werte ohne A/P-Suffix. Dieses Format muss beim Parsen speziell behandelt werden.

3. **Datums-Tippfehler:** Die Jahresverteilung zeigt Einträge weit ausserhalb des erwarteten Bereichs (z.B. 2000, 2012, 2027, 2052). Diese sollten gefiltert werden.

4. **Duplikate:** Im Sample wurden 403 Duplikate auf `Summons Number` gefunden. Eine Deduplizierung ist notwendig.

Diese Erkenntnisse fliessen direkt ins Pre-processing (`src/1_Pre_Processing/5.0_clean_parking_violations.ipynb`) ein.

In [11]:
spark.stop()